In [ ]:
import sqlite3
import pickle

import pandas as pd

In [ ]:
conn = sqlite3.connect("../data/raw/database_v2.db")

df = pd.read_sql_query(
    "SELECT * FROM player_stats",
    conn
)

conn.close()

In [ ]:
df.shape

In [ ]:
df["rank_tier"].value_counts().sort_index()

In [ ]:
df.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
model_df = df.copy()

model_df = model_df.drop(columns=[
    "replay_id",
    "rank",
    "team",
    "duration",
    "positioning_goals_against_while_last_defender"
])

model_df = model_df.dropna()

In [ ]:
model_df.shape

In [ ]:
model_df.isna().sum().sum()

In [ ]:
X = model_df.drop(columns=["rank_tier"])
y = model_df["rank_tier"]

In [ ]:
def tier_to_rank_group(tier):
    if tier <= 9:
        return "gold"
    elif tier <= 12:
        return "platinum"
    elif tier <= 15:
        return "diamond"
    elif tier <= 18:
        return "champion"
    else:
        return "grand_champion"

y_group = y.apply(tier_to_rank_group)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
def test_model(model, X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    return model, y_test, y_pred, accuracy

In [ ]:
rf_exact_tier_model = RandomForestClassifier(
    random_state=42
)

rf_exact_tier_model, rf_exact_y_test, rf_exact_y_pred, rf_exact_accuracy = test_model(
    rf_exact_tier_model,
    X,
    y
)

rf_exact_accuracy

In [ ]:
rf_exact_mae = abs(rf_exact_y_test - rf_exact_y_pred).mean()
rf_exact_within_1 = (abs(rf_exact_y_test - rf_exact_y_pred) <= 1).mean()
rf_exact_within_2 = (abs(rf_exact_y_test - rf_exact_y_pred) <= 2).mean()

rf_exact_mae, rf_exact_within_1, rf_exact_within_2

In [ ]:
rf_broad_rank_model = RandomForestClassifier(
    random_state=42
)

rf_broad_rank_model, rf_broad_y_test, rf_broad_y_pred, rf_broad_accuracy = test_model(
    rf_broad_rank_model,
    X,
    y_group
)

rf_broad_accuracy

In [ ]:
log_reg_broad_rank_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42))
])

log_reg_broad_rank_model, log_y_test, log_y_pred, log_accuracy = test_model(
    log_reg_broad_rank_model,
    X,
    y_group
)

log_accuracy

In [ ]:
with open("../model/log_reg_broad_rank_model_v2.pkl", "wb") as file:
    pickle.dump(log_reg_broad_rank_model, file)

In [ ]:
with open("../model/features_v2.pkl", "wb") as file:
    pickle.dump(list(X.columns), file)

## V2 Model Results

The v2 dataset contains 9000 player rows from 1500 ranked-standard 3v3 replays, with 100 replays per rank tier from Gold 1 through Grand Champion 3.

### Random Forest Exact-Tier Model

The exact-tier Random Forest model predicts one of 15 rank tiers.

Results:

- Exact tier accuracy: 22.1%
- Mean absolute tier error: 1.93 tiers
- Within 1 tier accuracy: 50.0%
- Within 2 tiers accuracy: 70.8%

Exact tier prediction is still difficult, but the model is usually close.

### Broad-Rank Models

The broad-rank models predict one of five rank groups:

- Gold
- Platinum
- Diamond
- Champion
- Grand Champion

Results:

- Random Forest broad-rank accuracy: 52.2%
- Logistic Regression broad-rank accuracy: 58.3%

The Logistic Regression broad-rank model performed best, so it was saved as `log_reg_broad_rank_model_v2.pkl`.

In [ ]:
import sqlite3, pandas as pd
conn = sqlite3.connect("../data/raw/database_v2.db")
df = pd.read_sql_query("SELECT * FROM player_stats LIMIT 1", conn)
conn.close()
print(df.columns.tolist())